# Notebook 36 — Corrected V4.2 Exact Serving Bundle

HNSW did not preserve ranking quality or materially improve latency. This notebook exports the accepted exact-GPU catalog in FP16 storage form, smoke-tests the immutable serving index, and uploads the large bundle directly to private Hugging Face storage. Locked Test is not used.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile
DRIVE_ROOT=Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT=DRIVE_ROOT/'data/gold/adaptive/v1'
BASE_GOLD_ROOT=DRIVE_ROOT/'data/gold/two_tower/v1_colab'
WORK_ROOT=Path('/content/recsys_exact_serving')
SOURCE_ROOT=WORK_ROOT/'runtime'; V4_DIR=WORK_ROOT/'v4'; BUNDLE_DIR=WORK_ROOT/'bundle'
REPORT_DIR=DRIVE_ROOT/'artifacts/adaptive/exact_serving_bundle_v1'
for p in (SOURCE_ROOT,V4_DIR,BUNDLE_DIR,REPORT_DIR): p.mkdir(parents=True,exist_ok=True)

In [ ]:
required={'torch':'torch','pandas':'pandas','pyarrow':'pyarrow','huggingface_hub':'huggingface_hub>=0.32'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import torch
assert torch.cuda.is_available(),'Select an L4 or A100 GPU.'
free=shutil.disk_usage('/content').free/1024**3
assert free>=12,f'Need at least 12 GiB local disk; found {free:.1f} GiB'
required_paths=[ADAPTIVE_ROOT/'sequence_examples/train',ADAPTIVE_ROOT/'sequence_examples/validation',BASE_GOLD_ROOT/'item_features/static',BASE_GOLD_ROOT/'transforms/numeric_stats.json']
assert all(p.exists() for p in required_paths),'Drive data is incomplete.'
print('GPU/data/disk preflight: PASS',torch.cuda.get_device_name(0),f'{free:.1f} GiB')

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
HF_REPO_ID='khoa1505/recsys-artifacts'
RUNTIME_FILE='runtime/corrected_v4_exact_serving_runtime_v1.zip'
RUNTIME_SHA256='167abf3d955d60320790480efc17207b03ce2e2cda003361e591575d33df5343'
V4_FILE='v4_corrected/v1/best_model.pt'
def token(name):
    try: raw=userdata.get(name)
    except Exception: raw=None
    if isinstance(raw,dict): raw=next((v for v in raw.values() if isinstance(v,str) and v.startswith('hf_')),None)
    if isinstance(raw,str) and raw.strip().startswith('hf_'): return raw.strip()
    return None
HF_TOKEN=token('HF_TOKEN') or getpass('Paste Hugging Face READ token: ').strip()
HF_WRITE_TOKEN=token('HF_WRITE_TOKEN') or getpass('Paste Hugging Face WRITE token: ').strip()
assert HF_TOKEN.startswith('hf_') and HF_WRITE_TOKEN.startswith('hf_')
runtime_zip=Path(hf_hub_download(HF_REPO_ID,RUNTIME_FILE,repo_type='model',token=HF_TOKEN))
v4=Path(hf_hub_download(HF_REPO_ID,V4_FILE,repo_type='model',token=HF_TOKEN))
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha(runtime_zip)==RUNTIME_SHA256 and v4.stat().st_size>100_000_000
target=V4_DIR/'best_model.pt'; target.unlink(missing_ok=True); target.symlink_to(v4.resolve())
if SOURCE_ROOT.exists(): shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as z: assert z.testzip() is None; z.extractall(SOURCE_ROOT)
BUILD_SCRIPT=SOURCE_ROOT/'scripts/build_corrected_v4_exact_serving_bundle.py'
assert BUILD_SCRIPT.exists(); compile(BUILD_SCRIPT.read_text(),str(BUILD_SCRIPT),'exec')
print('runtime/checkpoint integrity: PASS')

In [ ]:
shutil.rmtree(BUNDLE_DIR,ignore_errors=True); BUNDLE_DIR.mkdir(parents=True)
command=[sys.executable,str(BUILD_SCRIPT),'--adaptive-root',str(ADAPTIVE_ROOT),'--base-gold-root',str(BASE_GOLD_ROOT),'--corrected-v4-dir',str(V4_DIR),'--output-dir',str(BUNDLE_DIR),'--item-batch-size','8192','--device','cuda']
log_path=Path('/content/exact_serving_bundle.log')
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with log_path.open('w') as log:
    for line in process.stdout: print(line,end=''); log.write(line); log.flush()
code=process.wait()
if code!=0: raise RuntimeError('Bundle build failed:\n'+'\n'.join(log_path.read_text(errors='replace').splitlines()[-200:]))
manifest=json.loads((BUNDLE_DIR/'manifest.json').read_text())
assert manifest['candidate_items']==854893 and manifest['embedding_dim']==128
print('bundle build: PASS',sum(p.stat().st_size for p in BUNDLE_DIR.iterdir())/1024**2,'MiB')

In [ ]:
import numpy as np
sys.path.insert(0,str(SOURCE_ROOT))
from recommender.serving.exact_retrieval import ExactAdaptiveIndex
item_ids=np.load(BUNDLE_DIR/'item_ids.npy'); emb=np.load(BUNDLE_DIR/'item_embeddings_fp16.npy')
authors=np.load(BUNDLE_DIR/'author_ids.npy'); popularity=np.load(BUNDLE_DIR/'popularity.npy'); eligible=np.load(BUNDLE_DIR/'eligible_from_ms.npy')
engine=ExactAdaptiveIndex(item_ids,emb,authors,popularity,eligible,torch.device('cuda'))
result=engine.retrieve(emb[:2].astype('float32'),np.zeros((2,30),dtype='int64'),np.full(2,eligible.max()+1,dtype='int64'),[set(),set()],top_k=10)
assert result.item_ids.shape==(2,10) and (result.item_ids>=0).all() and (result.valid_counts==10).all()
print('exact serving smoke: PASS'); print(result.item_ids[:,:5])
del engine; torch.cuda.empty_cache()

In [ ]:
api=HfApi(token=HF_WRITE_TOKEN)
api.upload_folder(repo_id=HF_REPO_ID,repo_type='model',folder_path=str(BUNDLE_DIR),path_in_repo='serving/corrected_v4_2_exact_v1',commit_message='Upload Corrected V4.2 exact serving bundle')
for name in manifest['files']:
    assert api.file_exists(HF_REPO_ID,'serving/corrected_v4_2_exact_v1/'+name,repo_type='model')
assert api.file_exists(HF_REPO_ID,'serving/corrected_v4_2_exact_v1/manifest.json',repo_type='model')
shutil.copy2(BUNDLE_DIR/'manifest.json',REPORT_DIR/'manifest.json'); shutil.copy2(log_path,REPORT_DIR/log_path.name)
small=WORK_ROOT/'report'; shutil.rmtree(small,ignore_errors=True); small.mkdir(); shutil.copy2(BUNDLE_DIR/'manifest.json',small/'manifest.json'); shutil.copy2(log_path,small/log_path.name)
archive=Path(shutil.make_archive('/content/corrected_v4_exact_serving_bundle_report','zip',small))
print('Hugging Face upload verification: PASS'); files.download(str(archive))